In [1]:
!pip install ultralytics opencv-python numpy pandas matplotlib scipy filterpy
!pip install supervision

 done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached ultralytics_thop-2.0.18-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-2.2.6-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 182.7 kB/s  0:00:07 eta 0:00:01
Using cached numpy-2.2.6-cp313-cp313-macosx_14_0_arm64.whl (5.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 810.4/810.4 kB 345.9 kB/s  0:00:02eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 MB 127.6 kB/s  0:04:08m0:00:0100:08
Using cached ultralytics_thop-2.0.18-py3-none-any.whl (28 kB)
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110546 sha256=4d91b3b5f208dc242074ccae9ec102a8f5d617191cedcafab7e60d7d73b89246
  Stored in directory: /Users/akashgiri/Library/Caches/pip/wheels/79/33/43/53b597b8f63de80842202a5fed633eea6f5ce3e3f6c6efbab8
Successfully built filterpy
  Attempting uninstall: numpy━━━━━━━━━━━━━━

# Traffic Monitoring

In [11]:
# =============================================================================
# UNIFIED TRAFFIC MONITORING SYSTEM (JUPYTER VERSION - SINGLE OUTPUT VIDEO)
# Features:
# 1. Red Light Violation Detection
# 2. Vehicle Speed Estimation (km/h)
# 3. Congestion Detection (Stopped vehicles + avg speed)
# 4. Wrong Way Driving Alert
# 5. Smart Zoom (ANPR Ready)
# 6. Trajectory Tracking
# =============================================================================

import cv2
import numpy as np
import math
import os
from ultralytics import YOLO
import supervision as sv

# ────────────────────────────────────────────────
# 🔴 CONFIGURATION (EDIT THESE BASED ON YOUR VIDEO)
# ────────────────────────────────────────────────
VIDEO_PATH = "Traffic3.mp4"          # Your input video
OUTPUT_PATH = "unified_output.mp4"   # Final single output video

# Virtual traffic signal (for red-light logic)
RED_LIGHT = True  # Change to False if green signal

# Calibration (IMPORTANT – adjust after first run)
RED_LINE_Y = 300        # Stop line position (pixels)
WRONG_WAY_THRESHOLD = -25
ZOOM_TRIGGER_Y = 400    # Near camera region for zoom
PIXEL_TO_METER = 0.04   # Scale factor (affects speed accuracy)

print("Available Files:", os.listdir())

# ────────────────────────────────────────────────
# 🎯 LOAD YOLO MODEL (Vehicle Detection)
# ────────────────────────────────────────────────
print("Loading YOLOv8 model...")
model = YOLO("yolov8n.pt")  # Fast & suitable for real-time

# Vehicle classes (COCO dataset)
VEHICLE_CLASSES = [2, 3, 5, 7]  # car, motorcycle, bus, truck

# ────────────────────────────────────────────────
# 🧠 INITIALIZE TRACKER (ByteTrack - Industry Standard)
# ────────────────────────────────────────────────
tracker = sv.ByteTrack()

# ────────────────────────────────────────────────
# 🎥 VIDEO INITIALIZATION
# ────────────────────────────────────────────────
if not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(f"Video file not found: {VIDEO_PATH}")

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise IOError("Cannot open video. Check path or codec.")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS)) if cap.get(cv2.CAP_PROP_FPS) > 0 else 30

# Output video writer (SINGLE FINAL VIDEO)
out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

# ────────────────────────────────────────────────
# 📊 MEMORY STRUCTURES (Core Analytics)
# ────────────────────────────────────────────────
trajectories = {}        # Store path of each vehicle
violated_ids = set()     # Red-light violators
wrong_way_ids = set()    # Wrong direction vehicles
speed_memory = {}        # Speed per vehicle
stopped_ids = set()      # Stopped vehicles

frame_count = 0

print("\n🚀 Starting Unified Traffic Analytics...")
print("Press 'Q' to stop execution\n")

# =============================================================================
# 🔁 MAIN LOOP (ALL FEATURES IN ONE PIPELINE)
# =============================================================================
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("End of video reached")
        break

    frame_count += 1
    display_frame = frame.copy()

    # ─────────────────────────
    # 🚗 YOLO VEHICLE DETECTION
    # ─────────────────────────
    results = model(frame, conf=0.4, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)

    # Filter only vehicles
    if detections.class_id is not None:
        mask = np.isin(detections.class_id, VEHICLE_CLASSES)
        detections = detections[mask]

    # ─────────────────────────
    # 🧠 MULTI-OBJECT TRACKING
    # ─────────────────────────
    tracked = tracker.update_with_detections(detections)

    # Draw Red Light Stop Line
    cv2.line(display_frame, (0, RED_LINE_Y), (width, RED_LINE_Y), (0, 0, 255), 3)
    cv2.putText(display_frame, "STOP LINE (RED SIGNAL)", (20, RED_LINE_Y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    # ─────────────────────────
    # 🔍 PROCESS EACH VEHICLE
    # ─────────────────────────
    if tracked.tracker_id is not None:
        for bbox, track_id in zip(tracked.xyxy, tracked.tracker_id):
            x1, y1, x2, y2 = map(int, bbox)
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2

            # Store trajectory
            if track_id not in trajectories:
                trajectories[track_id] = []
            trajectories[track_id].append((cx, cy))

            # Draw trajectory path
            for i in range(1, len(trajectories[track_id])):
                cv2.line(display_frame,
                         trajectories[track_id][i - 1],
                         trajectories[track_id][i],
                         (255, 0, 0), 2)

            # 🚦 RED LIGHT VIOLATION
            if RED_LIGHT and cy > RED_LINE_Y:
                violated_ids.add(track_id)

            # 🚗 SPEED ESTIMATION
            speed_kmph = 0
            if len(trajectories[track_id]) >= 2:
                x_prev, y_prev = trajectories[track_id][-2]
                dist_pixels = math.hypot(cx - x_prev, cy - y_prev)
                speed_mps = (dist_pixels * PIXEL_TO_METER) * fps
                speed_kmph = speed_mps * 3.6
                speed_memory[track_id] = speed_kmph

                # 🛑 STOPPED VEHICLE (Congestion)
                if speed_kmph < 2:
                    stopped_ids.add(track_id)

            # 🚨 WRONG WAY DETECTION (Assuming downward traffic flow)
            if len(trajectories[track_id]) > 5:
                start_y = trajectories[track_id][0][1]
                direction = cy - start_y
                if direction < WRONG_WAY_THRESHOLD:
                    wrong_way_ids.add(track_id)

            # 🔍 SMART ZOOM (ANPR Preparation)
            if cy > ZOOM_TRIGGER_Y:
                crop = frame[y1:y2, x1:x2]
                if crop.size > 0:
                    zoom = cv2.resize(crop, (220, 140))
                    display_frame[10:150, width-230:width-10] = zoom
                    cv2.putText(display_frame, "AUTO ZOOM (Plate Zone)",
                                (width-260, 170),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0), 2)

            # 🎨 COLOR CODING BASED ON EVENTS
            color = (0, 255, 0)
            label = f"ID {track_id} | {int(speed_kmph)} km/h"

            if track_id in violated_ids:
                color = (0, 0, 255)
                label += " | RED VIOLATION"

            if track_id in wrong_way_ids:
                color = (255, 0, 255)
                label += " | WRONG WAY"

            if track_id in stopped_ids:
                color = (0, 255, 255)
                label += " | STOPPED"

            # Draw bounding box & label
            cv2.rectangle(display_frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(display_frame, label, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            cv2.circle(display_frame, (cx, cy), 4, (0, 0, 255), -1)

    # CONGESTION ANALYSIS (Average Speed)
    if len(speed_memory) > 0:
        avg_speed = sum(speed_memory.values()) / len(speed_memory)
        if avg_speed < 10:
            cv2.putText(display_frame, "CONGESTION DETECTED",
                        (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

    # DASHBOARD HUD
    cv2.putText(display_frame, f"Frame: {frame_count}", (20, height - 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.putText(display_frame, f"Vehicles Tracked: {len(trajectories)}",
                (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    cv2.putText(display_frame, f"Red Violations: {len(violated_ids)}",
                (20, 120),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    # Save ONE final output video
    out.write(display_frame)

    # Display in Jupyter/local
    cv2.imshow("Unified Traffic Analytics System", display_frame)
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break

# ────────────────────────────────────────────────
# CLEANUPq
# ────────────────────────────────────────────────
cap.release()
out.release()
cv2.destroyAllWindows()

print("\n Processing Completed Successfully!")
print(f" Single Output Video Saved As: {OUTPUT_PATH}")


Available Files: ['unified_output.mp4', 'yolov8n.pt', 'Traffic.ipynb', '.ipynb_checkpoints', 'Traffic1.mp4', 'unified_traffic_output.mp4', 'Traffic2.mp4', 'Traffic3.mp4']
Loading YOLOv8 model...

🚀 Starting Unified Traffic Analytics...
Press 'Q' to stop execution

End of video reached

✅ Processing Completed Successfully!
🎥 Single Output Video Saved As: unified_output.mp4
